In [ ]:
import sys
import os
import xarray as xr
import numpy as np
import pandas as pd
import cfgrib
import datetime
import copy
import sys
import gc
import salem
import geopandas as gpd
from scipy.optimize import curve_fit
import geopandas as gpd
from shapely.geometry import box
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import xarray as xr
from shapely.ops import transform
from functools import partial
import pyproj
import rioxarray
from shapely.geometry import mapping
from scipy.special import gamma
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import MinMaxScaler
import regionmask
import matplotlib.pyplot as plt
import xarray as xr
from xarray.backends import NetCDF4DataStore
from siphon.catalog import TDSCatalog
from datetime import datetime
from tqdm import tqdm
import geopandas as gp
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeat
from cartopy.mpl.ticker import LongitudeFormatter,LatitudeFormatter
from cartopy.io.shapereader import Reader, natural_earth
import matplotlib.pyplot as pltf
import matplotlib.ticker as mticker
from matplotlib.ticker import MaxNLocator  # 导入 MaxNLocator 类
from matplotlib.ticker import FormatStrFormatter
import dask
import dask.array as da
from cartopy.crs import epsg
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.ticker import MultipleLocator
%config InlineBackend.figure_formats = ['svg']
# 设置 Matplotlib 使用的字体
plt.rcParams['font.family'] = 'DejaVu Sans'  # 选择一个支持减号的字体

In [ ]:
ds = xr.open_dataset('data/MPDEV2018.nc')

In [ ]:
ds['Solar_LCOE'].plot()

In [ ]:
ds

In [ ]:
ec = np.sum(ds['chec2019']).values

In [ ]:
mapping = {
    'Wind_Power': ['Wind_LCOE', 'O_Wind_LCOE'],
    'Solar_Power': ['Solar_LCOE', 'O_Solar_LCOE'],
    'Wind_Solar_Power': ['Wind_Solar_LCOE', 'O_Wind_Solar_LCOE']
}

In [ ]:
# 使用来调整b的形状
ds['chec2019'] = ds['chec2019'].transpose('latitude', 'longitude', 'band')
ds['Wind_Solar_LCOE'] = ds['Wind_Solar_LCOE'].transpose('latitude', 'longitude', 'band')
ds['O_Wind_Solar_LCOE'] = ds['O_Wind_Solar_LCOE'].transpose('latitude', 'longitude', 'band')
ds['Solar_LCOE'] = ds['Solar_LCOE'].transpose('latitude', 'longitude', 'band')
ds['O_Solar_LCOE'] = ds['O_Solar_LCOE'].transpose('latitude', 'longitude', 'band')
ds['Wind_Solar_Power'] = ds['Wind_Power'] + ds['Solar_Power']
ds['chgriddis'] = ds['chgriddis'].transpose('latitude', 'longitude', 'band')
# 提取变量为NumPy数组，同时保留原始数据的位置信息
wind_power = (ds['Solar_Power']).values
wind_lcoe = ds['Solar_LCOE'].values
chec2019 = ds['chec2019'].values
chgriddis = ds['chgriddis'].values

# 找出非NaN值的索引
valid_indices = np.isfinite(wind_power) & np.isfinite(wind_lcoe) & np.isfinite(chgriddis) & np.isfinite(wind_lcoe) & np.isfinite(chec2019) & (wind_power != 0) & (wind_lcoe != 0)

# 使用非NaN值的索引提取数据
wind_power_valid = wind_power[valid_indices] * 1000 #kwh/grid
wind_lcoe_valid = wind_lcoe[valid_indices]  #CNY/kwh
chec2019_valid = chec2019[valid_indices] * 100 #kwh/grid
chdis_valid = chgriddis[valid_indices]

In [ ]:
##### import numpy as np
import random
import matplotlib.pyplot as plt
max_generations_list = [10,50,100,250,500]
# 定义问题参数
NUM_OBJECTIVES = 3  # 目标函数数量
POP_SIZE = pop_size = 200  # 种群大小
MAX_GENERATIONS =500
# 最大迭代次数
CROSSOVER_RATE = 0.4  # 交叉概率
MUTATION_RATE = 0.4  # 变异概率
DIMENSION = dimension = chec2019_valid.shape[0]  # 解的维度
# Q-learning parameters
alpha = 0.1  # 学习率
gamma = 0.9  # 折扣因子
epsilon = 0.1  # 探索率

# Q-learning更新函数
def q_learning_update(state, action, reward, next_state):
    current_q = q_table[state[0], state[1]]
    max_future_q = np.max(q_table[next_state[0], next_state[1]])
    new_q = (1 - alpha) * current_q + alpha * (reward + gamma * max_future_q)
    q_table[state[0], state[1]] = new_q

# 定义目标函数（示例）
def objective_function(solution):
    # 例如，假设你的目标函数是简单的二进制加法
    # 目标1: 统计solution中1的个数
    obj1 = func1(solution)
    # 目标2: 统计solution中0的个数
    obj2 = func2(solution)
    #
    obj3 = func3(solution)
    return [obj1, obj2, obj3]

def func1(solution):
    return np.sum(solution * wind_power_valid * 8760 )

def func2(solution):
    Q = np.where(wind_power_valid * 8760>chec2019_valid,(solution * (0.341 - 0.1 - wind_lcoe_valid)) * (wind_power_valid * 8760-chec2019_valid) + (solution * (0.341 - wind_lcoe_valid))*chec2019_valid,(solution * (0.341 - wind_lcoe_valid)) * (wind_power_valid * 8760))
    return np.sum(Q)

#越小越好
def func3(solution):
    return -np.sum(solution * chdis_valid)

# 初始化种群
def initialize_population(pop_size, dimension):
    return np.random.randint(2, size=(pop_size, dimension))

# 交叉操作
def crossover(parents, crossover_rate):
    children = np.zeros_like(parents)
    for i in range(len(parents)):
        idx1, idx2, idx3 = np.random.choice(len(parents), 3, replace=False)
        array1 = parents[idx2].astype(bool)
        array2 = parents[idx3].astype(bool)
        # 对两个数组进行异或计算
        result_XOR = np.logical_xor(array1, array2)
        crossover_mask = np.random.rand(dimension) < crossover_rate
        result_mask = np.logical_and(result_XOR, crossover_mask)
        x1 = parents[idx1].astype(bool)
        children_0 = np.where(crossover_mask,np.logical_and(result_XOR, x1) ,x1)
        children[i] = children_0.astype(int)
    return children

# 变异操作

def mutate(parent, children, mutation_rate):
    mutated_parent=np.copy(parent)
    mutated_children = np.copy(children)
    for i in range(len(mutated_children)):
        mutation_mask = np.random.rand(dimension) < mutation_rate
        mutated_children[i] = np.where(mutation_mask, parent[i], children[i])
    return mutated_children


# NSGA-II选择算法
def nsga2_selection(population, combined_population, pop_size):
    combined_fitness = np.array([objective_function(individual) for individual in combined_population])
    fronts = [[]]
    dominated_individuals = {}
    dominated_by_count = {}
    rank = np.zeros(len(combined_population))

    for i, ind in enumerate(combined_population):
        dominated_individuals[i] = []
        dominated_by_count[i] = 0
        for j, other_ind in enumerate(combined_population):
            if np.all(combined_fitness[j] <= combined_fitness[i]) and np.any(combined_fitness[j] < combined_fitness[i]):
                dominated_individuals[i].append(j)
            elif np.all(combined_fitness[i] <= combined_fitness[j]) and np.any(combined_fitness[i] < combined_fitness[j]):
                dominated_by_count[i] += 1
        if dominated_by_count[i] == 0:
            rank[i] = 0
            fronts[0].append(i)

    k = 0
    while len(fronts[k]) > 0:
        next_front = []
        for i in fronts[k]:
            for j in dominated_individuals[i]:
                dominated_by_count[j] -= 1
                if dominated_by_count[j] == 0:
                    rank[j] = k + 1
                    next_front.append(j)
        k += 1
        fronts.append(next_front)

    selected_population = []
    current_front = 0
    while len(selected_population) + len(fronts[current_front]) <= pop_size:
        selected_population.extend(fronts[current_front])
        current_front += 1
    
     # 计算拥挤距离
    crowding_distances = np.zeros(len(combined_population))
    for front in fronts:
        if len(front) > 0:
            for i in range(NUM_OBJECTIVES):
                objective_values = combined_fitness[front, i]
                sorted_indices = np.argsort(objective_values)
                crowding_distances[front[sorted_indices[0]]] = np.inf
                crowding_distances[front[sorted_indices[-1]]] = np.inf
                if objective_values[sorted_indices[-1]] == objective_values[sorted_indices[0]]:
                    continue
                for j in range(1, len(front) - 1):
                    crowding_distances[front[sorted_indices[j]]] += (
                        objective_values[sorted_indices[j + 1]] - objective_values[sorted_indices[j - 1]]) / (
                        objective_values[sorted_indices[-1]] - objective_values[sorted_indices[0]])

    # 根据拥挤距离选择额外的个体
    if len(selected_population) < pop_size:
        remaining_size = pop_size - len(selected_population)
        crowding_distances = crowding_distances[:len(fronts[current_front])]
        selected_indices = np.argsort(crowding_distances)[::-1][:remaining_size]
        selected_population.extend([fronts[current_front][idx] for idx in selected_indices])
    return [combined_population[idx] for idx in selected_population]

# 多目标优化的差分进化算法
def multi_objective_differential_evolution(pop_size, max_generations_list, dimension, crossover_rate, mutation_rate):
    population = initialize_population(pop_size, dimension)
    x = []
    y = []
    recorded_populations = []
    
    # 初始化Q值
    D0 = 0
    alpha = 0.1
    q_crossover = CROSSOVER_RATE
    q_mutation = MUTATION_RATE
    
    # 获取最大代数
    max_generations = max(max_generations_list)
    
    for gen in tqdm(range(max_generations), desc="Evolving", unit="generation"):
        combined_population = np.copy(population)
        crossover_rate = q_crossover
        mutation_rate = q_mutation
        #children = mutate(crossover(population, crossover_rate), mutation_rate)
        children = mutate(population, crossover(population, crossover_rate), mutation_rate)
        combined_population = np.vstack([combined_population, children])
        population = nsga2_selection(population, combined_population, pop_size)
        
        D = np.mean(np.abs(np.sum(population, axis=0) - POP_SIZE/2)/POP_SIZE/2)*100
        x.append(gen)
        y.append(D)
        
        # 检查当前代数是否需要记录种群
        if gen in max_generations_list:
            recorded_populations.append(np.copy(population))
        
        # 计算奖励
        #reward = 1 / (D + 1)
        
        # 更新Q值
        #epsilon = 0.01
        #q_crossover = q_crossover + alpha * (reward - q_crossover) + epsilon * (random.random() - 0.5)
        #q_mutation = q_mutation + alpha * (reward - q_mutation) + epsilon * (random.random() - 0.5)
        
        # 确保Q值在合理范围内
        #q_crossover = np.clip(q_crossover, 0.01, 1.0)
        #q_mutation = np.clip(q_mutation, 0.01, 1.0)
        
        # 逐渐减小学习率
        #alpha *= 0.99
        #epsilon *= 0.99
    
    return recorded_populations, x, y

In [ ]:
##### 新版遗传算法
# ======================
# 参数定义
# ======================
NUM_OBJECTIVES = 3
POP_SIZE = 200
MAX_GENERATIONS = 500
CROSSOVER_RATE = 0.4
MUTATION_RATE = 0.4
T = 20  # 邻域大小

# 假设输入数据（需要替换为实际的）
# wind_power_valid, chec2019_valid, wind_lcoe_valid, chdis_valid 均为 np.array
# dimension = len(chec2019_valid)
dimension = chec2019_valid.shape[0]


# ======================
# 目标函数定义
# ======================
def func1(solution):
    return np.sum(solution * wind_power_valid * 8760)

def func2(solution):
    Q = np.where(wind_power_valid * 8760 > chec2019_valid,
                 (solution * (0.341 - 0.1 - wind_lcoe_valid)) * (wind_power_valid * 8760 - chec2019_valid)
                 + (solution * (0.341 - wind_lcoe_valid)) * chec2019_valid,
                 (solution * (0.341 - wind_lcoe_valid)) * (wind_power_valid * 8760))
    return np.sum(Q)

def func3(solution):
    return -np.sum(solution * chdis_valid)

def objective_function(solution):
    return np.array([-func1(solution), -func2(solution), -func3(solution)])

# ======================
# 初始化种群
# ======================
def initialize_population(pop_size, dimension):
    return np.random.randint(2, size=(pop_size, dimension))

# ======================
# 差分进化（与 MOEA/D 一致）
# ======================
def differential_evolution_nsga(population, i, crossover_rate, mutation_rate):
    idxs = list(range(len(population)))
    idxs.remove(i)
    a, b, c = population[np.random.choice(idxs, 3, replace=False)]
    mutant = np.clip(a + mutation_rate * (b - c), 0, 1)
    crossover_mask = np.random.rand(len(mutant)) < crossover_rate
    offspring = np.where(crossover_mask, mutant, population[i])
    return np.round(offspring).astype(int)

# ======================
# NSGA-II 辅助函数
# ======================
def dominates(ind1, ind2):
    """判断个体 ind1 是否支配 ind2"""
    return np.all(ind1 <= ind2) and np.any(ind1 < ind2)

def fast_non_dominated_sort(fitness):
    S = [[] for _ in range(len(fitness))]
    n = np.zeros(len(fitness))
    rank = np.zeros(len(fitness))
    fronts = [[]]

    for p in range(len(fitness)):
        for q in range(len(fitness)):
            if dominates(fitness[p], fitness[q]):
                S[p].append(q)
            elif dominates(fitness[q], fitness[p]):
                n[p] += 1
        if n[p] == 0:
            rank[p] = 0
            fronts[0].append(p)

    i = 0
    while len(fronts[i]) > 0:
        Q = []
        for p in fronts[i]:
            for q in S[p]:
                n[q] -= 1
                if n[q] == 0:
                    rank[q] = i + 1
                    Q.append(q)
        i += 1
        fronts.append(Q)
    fronts.pop()
    return fronts

def crowding_distance(fitness, front):
    distance = np.zeros(len(front))
    if len(front) == 0:
        return distance
    F = np.array([fitness[i] for i in front])
    for m in range(F.shape[1]):
        sorted_idx = np.argsort(F[:, m])
        distance[sorted_idx[0]] = distance[sorted_idx[-1]] = np.inf
        f_min, f_max = F[sorted_idx[0], m], F[sorted_idx[-1], m]
        if f_max - f_min == 0:
            continue
        for i in range(1, len(front) - 1):
            distance[sorted_idx[i]] += (F[sorted_idx[i + 1], m] - F[sorted_idx[i - 1], m]) / (f_max - f_min)
    return distance

# ======================
# NSGA-II 主算法
# ======================
def nsga2(pop_size, dimension, num_objectives, max_generations, crossover_rate, mutation_rate):
    population = initialize_population(pop_size, dimension)
    fitness = np.array([objective_function(ind) for ind in population])

    history = []
    x, y = [], []

    for gen in tqdm(range(max_generations), desc="NSGA-II Evolution", unit="gen"):
        # 生成子代
        offspring = []
        for i in range(pop_size):
            child = differential_evolution_nsga(population, i, crossover_rate, mutation_rate)
            offspring.append(child)
        offspring = np.array(offspring)
        offspring_fitness = np.array([objective_function(ind) for ind in offspring])

        # 合并父代与子代
        combined_pop = np.vstack((population, offspring))
        combined_fit = np.vstack((fitness, offspring_fitness))

        # 快速非支配排序
        fronts = fast_non_dominated_sort(combined_fit)
        new_population = []
        new_fitness = []

        for front in fronts:
            if len(new_population) + len(front) > pop_size:
                dist = crowding_distance(combined_fit, front)
                sorted_idx = np.argsort(-dist)
                for i in sorted_idx:
                    if len(new_population) < pop_size:
                        new_population.append(combined_pop[front[i]])
                        new_fitness.append(combined_fit[front[i]])
                    else:
                        break
                break
            else:
                for i in front:
                    new_population.append(combined_pop[i])
                    new_fitness.append(combined_fit[i])

        population = np.array(new_population)
        fitness = np.array(new_fitness)

        if gen % 50 == 0 or gen == max_generations - 1:
            history.append(population.copy())

        D = np.mean(np.abs(np.sum(population, axis=0) - POP_SIZE / 2) / (POP_SIZE / 2)) * 100
        x.append(gen)
        y.append(D)

    return population, fitness, history, x, y

In [ ]:
import numpy as np
import random
from tqdm import tqdm
import matplotlib.pyplot as plt



# 假设输入数据（需要替换为实际的）
# wind_power_valid, chec2019_valid, wind_lcoe_valid, chdis_valid 均为 np.array
# dimension = len(chec2019_valid)
dimension = chec2019_valid.shape[0]





# ======================
# 初始化权重向量与邻域
# ======================
def init_weight_vectors(pop_size, num_objectives, T):
    weight_vectors = np.random.rand(pop_size, num_objectives)
    weight_vectors /= np.sum(weight_vectors, axis=1, keepdims=True)
    distances = np.linalg.norm(weight_vectors[:, None, :] - weight_vectors[None, :, :], axis=2)
    B = np.argsort(distances, axis=1)[:, :T]
    return weight_vectors, B


# ======================
# 初始化种群
# ======================
def initialize_population(pop_size, dimension):
    return np.random.randint(2, size=(pop_size, dimension))


# ======================
# 交叉与变异
# ======================
def differential_evolution(population, i, B, crossover_rate, mutation_rate):
    idxs = B[i]
    a, b, c = population[np.random.choice(idxs, 3, replace=False)]
    mutant = np.clip(a + mutation_rate * (b - c), 0, 1)
    crossover_mask = np.random.rand(len(mutant)) < crossover_rate
    offspring = np.where(crossover_mask, mutant, population[i])
    return np.round(offspring).astype(int)


# ======================
# 聚合函数 (Tchebycheff)
# ======================
def tchebycheff(f, z, weight):
    return np.max(weight * np.abs(f - z))


# ======================
# MOEA/D 主算法
# ======================
def moead(pop_size, dimension, num_objectives, max_generations, crossover_rate, mutation_rate, T):
    # 初始化
    population = initialize_population(pop_size, dimension)
    weight_vectors, B = init_weight_vectors(pop_size, num_objectives, T)
    fitness = np.array([objective_function(ind) for ind in population])
    z = np.min(fitness, axis=0)
    x = []
    y = []
    history = []

    for gen in tqdm(range(max_generations), desc="MOEA/D Evolution", unit="gen"):
        for i in range(pop_size):
            # 生成新个体
            offspring = differential_evolution(population, i, B, crossover_rate, mutation_rate)
            f_offspring = objective_function(offspring)

            # 更新参考点
            z = np.minimum(z, f_offspring)

            # 更新邻域
            for j in B[i]:
                f_neighbor = fitness[j]
                g_old = tchebycheff(f_neighbor, z, weight_vectors[j])
                g_new = tchebycheff(f_offspring, z, weight_vectors[j])
                if g_new < g_old:
                    population[j] = offspring
                    fitness[j] = f_offspring

        if gen % 50 == 0 or gen == max_generations - 1:
            history.append(population.copy())
        D = np.mean(np.abs(np.sum(population, axis=0) - POP_SIZE/2)/POP_SIZE/2)*100
        x.append(gen)
        y.append(D)

    return population, fitness, history,x,y


In [ ]:
import numpy as np
import random
from tqdm import tqdm
import matplotlib.pyplot as plt

# ---------- 你已有的全局变量/函数（示例）
# objective_function(solution) 已在你的环境中定义（返回 np.array([f1,f2,f3])）
# dimension = chec2019_valid.shape[0]

# ---------- 差分进化算子（保留你给的版本，返回浮点 -> caller 可二值化/四舍五入）
def differential_evolution(population, i, B, crossover_rate, mutation_rate):
    idxs = B[i]
    a, b, c = population[np.random.choice(idxs, 3, replace=False)]
    mutant = np.clip(a + mutation_rate * (b - c), 0, 1)
    crossover_mask = np.random.rand(len(mutant)) < crossover_rate
    offspring = np.where(crossover_mask, mutant, population[i])
    return np.round(offspring).astype(int)


# ---------- 支配关系判断
def dominates(a, b):
    # a, b: 两个目标向量（numpy array），更小更好
    return np.all(a <= b) and np.any(a < b)
    #return np.all(a >= b) and np.any(a > b)


# ---------- 计算 Strength（每个解支配的数量）
def compute_strengths(F):
    N = len(F)
    S = np.zeros(N, dtype=float)
    for i in range(N):
        for j in range(N):
            if dominates(F[i], F[j]):
                S[i] += 1.0
    return S


# ---------- 计算 Raw fitness（被谁支配 -> 强度之和）
def compute_raw_fitness(F, strengths):
    N = len(F)
    R = np.zeros(N, dtype=float)
    for i in range(N):
        for j in range(N):
            if dominates(F[j], F[i]):  # j 支配 i
                R[i] += strengths[j]
    return R


# ---------- 密度估计（基于 kth 最近邻距离）返回 density 值
def compute_density(F, k=None):
    N = len(F)
    if N == 0:
        return np.array([])
    # 距离矩阵
    dist = np.linalg.norm(F[:, None, :] - F[None, :, :], axis=2)
    np.fill_diagonal(dist, np.inf)
    if k is None:
        k = int(np.sqrt(N))
        if k < 1:
            k = 1
    # kth 最近距离
    kth = np.sort(dist, axis=1)[:, k-1]  # k-1 因为索引从0开始
    density = 1.0 / (kth + 2.0)
    return density


# ---------- 计算 SPEA2 综合适应度
def compute_spea2_fitness(F):
    strengths = compute_strengths(F)
    raw = compute_raw_fitness(F, strengths)
    density = compute_density(F, k=max(1, int(np.sqrt(len(F)))))
    return raw + density, strengths, raw, density


# ---------- 环境选择（选择 archive_size 个解）
# 如果候选数 > archive_size，使用截断策略：重复删除距离最小的个体，直到满足大小
def environmental_selection(combined_pop, combined_F, archive_size):
    # combined_pop: (N, dim), combined_F: (N, num_obj)
    
    fitness, strengths, raw, density = compute_spea2_fitness(combined_F)
    idx_sorted = np.argsort(fitness)  # 从小到大（较优）
    selected = list(idx_sorted[:archive_size]) if len(idx_sorted) >= archive_size else list(idx_sorted)

    # 如果不足 archive_size，直接把最优的一些放入（已有）
    if len(selected) == archive_size:

        return combined_pop[selected], combined_F[selected]

    # 若候选 < archive_size（不常见），补齐最优解
    if len(selected) < archive_size:
        remaining = [i for i in range(len(combined_pop)) if i not in selected]
        needed = archive_size - len(selected)
        # 按 fitness 继续补齐
        remaining_sorted = sorted(remaining, key=lambda x: fitness[x])
        selected.extend(remaining_sorted[:needed])

        return combined_pop[selected], combined_F[selected]

    # 当候选 > archive_size（需截断）
    # 取初始候选（所有 fitness 最小的 archive_size 以外则会被考虑删除）
    selected = list(idx_sorted[:archive_size])
    # If more than archive_size due to ties (rare), we truncate by distance
    while len(selected) > archive_size:

        # 计算候选集中目标空间的距离矩阵
        selF = combined_F[selected]
        dist = np.linalg.norm(selF[:, None, :] - selF[None, :, :], axis=2)
        np.fill_diagonal(dist, np.inf)
        # 找到每个解到最近邻的距离
        nearest = np.min(dist, axis=1)
        # 要删除的索引是最近邻距离最小的解（即最拥挤）
        remove_idx = np.argmin(nearest)
        del selected[remove_idx]
    return combined_pop[selected], combined_F[selected]


# ---------- 主 SPEA2 + DE 算法
def SPEA2_with_DE(
    pop_size=POP_SIZE,
    archive_size=None,
    dim=None,
    generations=MAX_GENERATIONS,
    crossover_rate=CROSSOVER_RATE,
    mutation_rate=MUTATION_RATE,
    verbose=True
):
    if dim is None:
        raise ValueError("请传入 dim（决策变量维度），例如 dimension = chec2019_valid.shape[0]")
    if archive_size is None:
        archive_size = pop_size

    # 初始化种群（二进制）
    population = np.random.randint(2, size=(pop_size, dim))
    # 初始档案为空
    archive = np.empty((0, dim), dtype=int)
    archive_F = np.empty((0, NUM_OBJECTIVES), dtype=float)

    x = []
    y = []
    history = []
    for gen in tqdm(range(generations), desc="SPEA2+DE", unit="gen"):
    #for gen in range(generations):
        # 计算当前种群目标

        pop_F = np.array([objective_function(ind) for ind in population])


        # 合并种群与档案进行适应度分配与环境选择
        if archive.shape[0] > 0:
            combined_pop = np.vstack([population, archive])
            combined_F = np.vstack([pop_F, archive_F])
        else:
            combined_pop = population.copy()
            combined_F = pop_F.copy()


        # 环境选择：选择新的 archive
        new_archive, new_archive_F = environmental_selection(combined_pop, combined_F, archive_size)
        archive = new_archive.copy()
        archive_F = new_archive_F.copy()

        # 从 archive 中采样父代生成子代（若 archive 为空则从 population 中采样）
        parent_pool = archive if archive.shape[0] > 0 else population
        # 若 parent_pool 小于 pop_size，允许重复采样
        parents = parent_pool[np.random.choice(len(parent_pool), pop_size, replace=True)]

        # 为 DE 构建邻域 B（这里简单使用“除自身外所有索引”）
        B = [np.delete(np.arange(len(parents)), i) for i in range(len(parents))]
        max_generations = MAX_GENERATIONS
        if gen % 50 == 0 or gen == max_generations - 1:
            history.append(archive.copy())
        D = np.mean(np.abs(np.sum(archive, axis=0) - POP_SIZE/2)/POP_SIZE/2)*100
        x.append(gen)
        y.append(D)

        # 用差分进化算子产生下一代
        new_pop = []
        for i in range(len(parents)):
            # differential_evolution 接受浮点/二进制混合，我们已设 parents 为 0/1，所以算子内会产生浮点 mutant，再 round
            offspring = differential_evolution(parents, i, B, crossover_rate, mutation_rate)
            # offspring 已经被 round 为 0/1
            new_pop.append(offspring)
        population = np.array(new_pop, dtype=int)


    # 返回最终档案与目标值（非支配集近似）
    return archive, archive_F,history, x, y

In [ ]:
import numpy as np
import netCDF4 as nc
import pandas as pd
from pathlib import Path
def save_population_to_nc(population, filename):
    """将种群保存为NetCDF文件"""
    # 创建NetCDF文件
    with nc.Dataset(filename, 'w', format='NETCDF4') as ds:
        # 定义维度
        n_individuals = len(population)
        n_genes = len(population[0])
        n_seed = len(population[0][0])
        
        ds.createDimension('individuals', n_individuals)
        ds.createDimension('genes', n_genes)
        ds.createDimension('seed',n_seed)
        
        # 添加变量
        pop_var = ds.createVariable('population', 'i4', ('individuals', 'genes','seed'))
        pop_var[:] = population
        
        # 添加属性说明
        ds.description = f"Optimized wind farm layout (size={n_individuals}x{n_genes}x{n_seed})"
        #ds.algorithm = "QL-NSGA-II" if "Q2" in filename else "MOEA/D"

def save_convergence_to_excel(data_dict, filename):
    """将收敛数据保存为Excel"""
    df = pd.DataFrame(data_dict)
    df.to_excel(filename, index=False, engine='openpyxl')

In [ ]:
for page in range(0,10):
    for upper, lower_list in mapping.items():
            for lower in lower_list:            
                # 获取 x 和 y 的值
                wind_power = ds[upper].values
                wind_lcoe = ds[lower].values
                valid_indices = np.isfinite(wind_power) & np.isfinite(wind_lcoe) & np.isfinite(chec2019) & (wind_power != 0) & (wind_lcoe != 0)
                # 使用非NaN值的索引提取数据
                wind_power_valid = wind_power[valid_indices] * 1000 #kwh/grid
                wind_lcoe_valid = wind_lcoe[valid_indices]  #CNY/kwh
                chec2019_valid = chec2019[valid_indices] * 100 #kwh/grid
                chdis_valid = chgriddis[valid_indices]
                DIMENSION = dimension = chec2019_valid.shape[0]  # 解的维度
                #final_population_Q2,x_Q2,y_Q2 = multi_objective_differential_evolution(POP_SIZE, max_generations_list, DIMENSION, CROSSOVER_RATE, MUTATION_RATE)
                final_pop_moea, final_fit_moea, history_moea,x_MOEA,y_MOEA = moead(pop_size=POP_SIZE,dimension=dimension,num_objectives=NUM_OBJECTIVES,max_generations=MAX_GENERATIONS,crossover_rate=CROSSOVER_RATE,mutation_rate=MUTATION_RATE,T=T)
                final_population_Q2, fitness_Q2, history_Q2, x_Q2,y_Q2 = nsga2(POP_SIZE, dimension, NUM_OBJECTIVES,MAX_GENERATIONS, CROSSOVER_RATE, MUTATION_RATE)
                #final_population_MOEA,convergence_curve,x_MOEA,y_MOEA=MOEA_D(POP_SIZE, max_generations_list, DIMENSION, CROSSOVER_RATE, MUTATION_RATE)
                final_population_S2,ob_s2,history_S2,x_S2,y_S2 = SPEA2_with_DE(pop_size=POP_SIZE, archive_size=POP_SIZE, dim=dimension, generations=MAX_GENERATIONS)
                # 准备数据
                q2_data = {
                    'Generation': x_Q2,
                    'D_Indicator': y_Q2
                }
                moea_data = {
                    'Generation': x_MOEA,
                    #'Convergence': convergence_curve,
                    'D_Indicator': y_MOEA
                }
                s2_data = {
                    'Generation': x_S2,
                    'D_Indicator': y_S2
                }
                # 保存种群数据
                save_population_to_nc(history_Q2, f"result_0.4_0.341_2018/{page}/{lower}_NC_Q2.nc")
                save_population_to_nc(history_moea, f"result_0.4_0.341_2018/{page}/{lower}_NC_MOEA.nc")
                save_population_to_nc(history_S2, f"result_0.4_0.341_2018/{page}/{lower}_NC_S2.nc")
                # 保存Excel
                save_convergence_to_excel(q2_data, f"result_0.4_0.341_2018/{page}/{lower}_EX_Q2.xlsx")
                save_convergence_to_excel(moea_data, f"result_0.4_0.341_2018/{page}/{lower}_EX_MOEA.xlsx")
                save_convergence_to_excel(s2_data, f"result_0.4_0.341_2018/{page}/{lower}_EX_S2.xlsx")

In [ ]:
MM

In [ ]:
# 绘制帕累托前沿图
def plot_pareto_front(final_population,ax):
    objectives = np.array([objective_function(individual) for individual in final_population])
    obj1_values = objectives[:, 0]/(10 ** 12)
    obj2_values = objectives[:, 1]/(10 ** 11)
    obj3_values = -objectives[:, 2]/(10 ** 8)

    ax.scatter3D(obj1_values, obj2_values, obj3_values, color='blue', alpha=0.7)
    ax.set_xlabel('Energy Production($\\times10^{12}$ MWh)')
    ax.set_ylabel('Economic Benefit($\\times10^{11}$ CNY)')
    ax.set_zlabel('Sum of Distance($\\times10^{8}$ m)')
    #ax.set_title('Pareto Front')
    ax.grid(True)

In [ ]:
# 修改后的 plot_line_chart 函数
def plot_line_chart(x, y, ax):
    ax.plot(x, y)
    ax.set_xlim(0, 500)
    ax.set_xlabel('Generation')
    ax.set_ylabel('Indicator %')
    ax.grid(True)

In [ ]:
def plot_hist_chart(final_population, ax):
    # 给定的数组D
    D = np.sum(final_population,axis=0)
    # 创建一个与valid_indices相同长度的数组，用于索引D中的值
    # 由于D的长度小于valid_indices，我们需要重复D中的值
    condition = valid_indices.ravel()
    # 创建一个全0的数组，长度与condition相同
    #result = np.zeros_like(condition, dtype=int)
    result = np.full_like(condition, np.nan, dtype=int)
    # 我们可以通过计算valid_indices中True的数量来确定需要重复的次数

    # 计算condition中True的数量
    true_count = np.sum(condition) 
    # 确保D数组有足够的元素来填充True的位置
    if true_count > len(D):
        raise ValueError("D数组中没有足够的元素来填充所有的True位置")
    # 使用numpy的掩码操作来填充True位置
    result[condition] = D[:true_count]
    data = result.reshape(140,309,1)
    D_da = xr.DataArray(
        data=data,
        dims=ds['Wind_Power'].dims,
        coords=ds['Wind_Power'].coords
    )
    P_da = D_da.drop_vars(['band', 'number', 'surface'])
    # 使用xarray的直方图绘制功能，并且仅绘制非NaN值
    xr.where(P_da>=0,P_da,np.nan).plot.hist(ax=ax,range=(0, 200))
    #ax.set_title('Histogram of Count')
    ax.set_xlabel('Count')
    ax.set_ylabel('Selected Number')

In [ ]:
def plot_map(final_population, ax):
     # 给定的数组D
    D = np.sum(final_population,axis=0)
    # 创建一个与valid_indices相同长度的数组，用于索引D中的值
    # 由于D的长度小于valid_indices，我们需要重复D中的值
    condition = valid_indices.ravel()
    # 创建一个全0的数组，长度与condition相同
    #result = np.zeros_like(condition, dtype=int)
    result = np.full_like(condition, np.nan, dtype=int)
    # 我们可以通过计算valid_indices中True的数量来确定需要重复的次数

    # 计算condition中True的数量
    true_count = np.sum(condition) 
    # 确保D数组有足够的元素来填充True的位置
    if true_count > len(D):
        raise ValueError("D数组中没有足够的元素来填充所有的True位置")
    # 使用numpy的掩码操作来填充True位置
    result[condition] = D[:true_count]
    data = result.reshape(140,309,1)
    D_da = xr.DataArray(
        data=data,
        dims=ds['Wind_Power'].dims,
        coords=ds['Wind_Power'].coords
    )
    P_da = D_da.drop_vars(['band', 'number', 'surface'])
    # 使用xarray的直方图绘制功能，并且仅绘制非NaN值
    # 你需要确保final_population的维度与axs2的子图对应
    
    # 绘制地理图

    i=0
    x1=[]
    y1=[]
    y2=[]
    y3=[]
    for i in range(0,200):
        T = xr.where(P_da>=i,P_da,np.nan)
        x1.append(i)
        y1.append(np.sum(T * wind_power*1000 * 8760))
        y2.append(np.sum(xr.where(wind_power * 8760>chec2019,(T * (0.241 - wind_lcoe)) * (wind_power * 8760-chec2019) + (T * (0.307 - wind_lcoe))*chec2019,(T * (0.307 - wind_lcoe)) * (wind_power * 8760))))
        y3.append(np.sum(T * chgriddis))
    # 设置地理图的范围和投影

    ax.add_feature(provinces, linewidth=0.6, zorder=2)
    # 加载分辨率为50的海岸线
    ax.add_feature(cfeat.COASTLINE.with_scale('50m'), linewidth=0.6, zorder=10)
    # 加载分辨率为50的河流~
    ax.add_feature(cfeat.RIVERS.with_scale('50m'), zorder=10)
     # 加载分辨率为50的湖泊
    ax.add_feature(cfeat.LAKES.with_scale('50m'), zorder=10)
     # --设置网格属性

    ax.set_xticks(np.arange(70, 110 + 5, 5))
    ax.set_yticks(np.arange(20, 45 + 5, 5))
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    #ax.xaxis.set_minor_locator(MultipleLocator(1))
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    #ax.yaxis.set_minor_locator(MultipleLocator(1))

    
    gl = ax.gridlines(
        crs = ccrs.PlateCarree(),
        draw_labels = False,
        linewidth = 0.9,
        color = 'k',
        alpha = 0.5,
        linestyle = '--',
    )
    # 关闭顶部x轴和右侧y轴的标签
# 关闭上下左右所有标签
    gl.xlabels_top = False
    gl.xlabels_bottom = False
    gl.ylabels_left = False
    gl.ylabels_right = False

    ax.tick_params(axis='both', labelsize=5, direction='out')

    # 关闭顶部和右侧的刻度文本
    ax.tick_params(top=False, right=False, labeltop=False, labelright=False)
    # --设置刻度

    xr.where(P_da>=0,P_da,np.nan).plot(ax=ax, transform=ccrs.PlateCarree(), 
                        cmap='coolwarm', add_colorbar=False, vmin=0, vmax=200)
    # -- 设置范围
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_extent([70, 110, 20, 45], crs=ccrs.PlateCarree())
    return P_da,x1,y1,y2  # 返回绘制对象，以便稍后可以用于创建colorbar

In [ ]:
# 创建一个组图，行数为映射字典中所有列表长度的总和，列数为3
# 创建一个组图，行数为映射字典中所有列表长度的总和，列数为3
# 创建一个图形
fig = plt.figure(figsize=(30, 30))

# 创建一个 5x6 的 3D 子图网格
axs_3d = np.empty((5, 6), dtype=object)
for i in range(5):
    for j in range(6):
        axs_3d[i, j] = fig.add_subplot(6, 6, i * 6 + j + 1, projection='3d')

# 创建一个 1x6 的 2D 子图网格
axs_2d = np.empty((1, 6), dtype=object)
for j in range(6):
    axs_2d[0, j] = fig.add_subplot(6, 6, 5 * 6 + j + 1)
fig2, axs2 = plt.subplots(5, 6, figsize=(25, 12),gridspec_kw={'right': 0.95},
                          subplot_kw={'projection': ccrs.PlateCarree()})
# 创建一个映射对象列表，用于创建colorbar

# 设置colorbar的范围
vmin, vmax = 0, 200

# 定义颜色条的轴位置
cax = fig2.add_axes([0.95, 0.12, 0.01, 0.76])  # left, bottom, width, height
cbar = fig2.colorbar(
    plt.cm.ScalarMappable(norm=plt.Normalize(vmin=vmin, vmax=vmax), cmap='coolwarm'),
    cax=cax,  # 使用专门的颜色条轴
    orientation='vertical'
)

provinces = cfeat.ShapelyFeature(
        Reader('/kaggle/input/province/Province_9/Province_9.shp').geometries(),
        ccrs.PlateCarree() , edgecolor='k',
        facecolor='none'
)

###记录结果的数列
dataarrays = []
# 初始化一个空的Dataset
combined_dataset = xr.Dataset()
data_array=[]
pcm_list = []
# 用于跟踪当前 subplot 的索引
Max_G = [10,50,100,250,500]
col_index = 0
row_index = 0
levels = np.linspace(0, 100, 30)
# 遍历映射字典
for max_generations in Max_G:
    col_index=0
    for upper, lower_list in mapping.items():
        for lower in lower_list:            
            # 获取 x 和 y 的值
            wind_power = ds[upper].values
            wind_lcoe = ds[lower].values
            # 运行算法并绘制帕累托前沿图
            # 找出非NaN值的索引
            valid_indices = np.isfinite(wind_power) & np.isfinite(wind_lcoe) & np.isfinite(chec2019) & (wind_power != 0) & (wind_lcoe != 0)
            # 使用非NaN值的索引提取数据
            wind_power_valid = wind_power[valid_indices] * 1000 #kwh/grid
            wind_lcoe_valid = wind_lcoe[valid_indices]  #CNY/kwh
            chec2019_valid = chec2019[valid_indices] * 100 #kwh/grid
            chdis_valid = chgriddis[valid_indices]
            DIMENSION = dimension = chec2019_valid.shape[0]  # 解的维度
            #def func1(solution):
                #return np.sum(solution * wind_power_valid * 8760 )
            #def func2(solution):
                #Q = np.where(wind_power_valid * 8760>chec2019_valid,(solution * (0.241 - wind_lcoe_valid)) * (wind_power_valid * 8760-chec2019_valid) + (solution * (0.307 - wind_lcoe_valid))*chec2019_valid,(solution * (0.307 - wind_lcoe_valid)) * (wind_power_valid * 8760))
                #return np.sum(Q)
            final_population,x,y = multi_objective_differential_evolution(POP_SIZE, max_generations, DIMENSION, CROSSOVER_RATE, MUTATION_RATE)
            # 获取子图的行和列索引
            #row_index = subplot_index // 2
            #col_index = subplot_index % 2
            plot_pareto_front(final_population,axs_3d[row_index][col_index])
            # 绘制折线图
            ax_hist = axs_3d[row_index][col_index].inset_axes([0.6, 0.6, 0.35, 0.35])
            plot_hist_chart(final_population, ax_hist)
            data_array = plot_map(final_population, axs2[row_index][col_index])
            # 将生成的DataArray添加到列表中
            data_array[0].name = f'{lower}_{max_generations}_{max_generations}'
            combined_dataset[data_array[0].name] = data_array[0] 
            #pcm_list.append(pcm)  # 将绘制对象添加到列表中
            if max_generations == 500:
                plot_line_chart(x , y, axs_2d[0][col_index])
                   # 将生成的DataArray添加到Dataset中              
            col_index += 1
    row_index += 1
# 调整布局并显示组图
# 创建一个全局colorbar
#cax = fig2.add_axes([1, 0.1, 0.02, 0.8])  # [left, bottom, width, height]
# 使用列表中的第一个绘制对象创建colorbar，并设置范围
#cbar = fig2.colorbar(pcm_list[0], cax=cax, extend='max')  # 设置colorbar的范围为0到200
#fig2.colorbar
#cbar.set_label('Count')  # 设置colorbar的标签
# 如果需要，可以将合并后的Dataset保存到NetCDF文件
combined_dataset.to_netcdf('/kaggle/working/Result/Result_1000_dataset_3.nc')
plt.tight_layout()
# 保存第一个图形
fig.savefig('/kaggle/working/Result/figure1_3.png')
fig2.savefig('/kaggle/working/Result/figure2_3.png',dpi=450)

In [ ]:
combined_dataset.to_netcdf('/kaggle/working/Result_1000_dataset_4.nc')
plt.tight_layout()
# 保存第一个图形
fig.savefig('/kaggle/working/figure1_4.png')
fig2.savefig('/kaggle/working/figure2_4.png',dpi=450)

In [ ]:
# 创建一个组图，行数为映射字典中所有列表长度的总和，列数为3
# 创建一个组图，行数为映射字典中所有列表长度的总和，列数为3
fig, axs = plt.subplots(6, 6, figsize=(30, 28))
#fig2, axs2 = plt.subplots(5, 6, figsize=(30, 28))
# 用于跟踪当前 subplot 的索引
Max_G = [10,50,100,200,500]
col_index = 0
row_index = 0
# 遍历映射字典
for max_generations in Max_G:
    col_index=0
    for upper, lower_list in mapping.items():
        for lower in lower_list:            
            # 获取 x 和 y 的值
            wind_power = ds[upper].values
            wind_lcoe = ds[lower].values
            # 运行算法并绘制帕累托前沿图
            # 找出非NaN值的索引
            valid_indices = np.isfinite(wind_power) & np.isfinite(wind_lcoe) & np.isfinite(chec2019) & (wind_power != 0) & (wind_lcoe != 0)
            # 使用非NaN值的索引提取数据
            wind_power_valid = wind_power[valid_indices] * 1000 #kwh/grid
            wind_lcoe_valid = wind_lcoe[valid_indices]  #CNY/kwh
            chec2019_valid = chec2019[valid_indices] * 100 #kwh/grid
            DIMENSION = dimension = chec2019_valid.shape[0]  # 解的维度
            def func1(solution):
                return np.sum(solution * wind_power_valid * 8760 )
            def func2(solution):
                Q = np.where(wind_power_valid * 8760>chec2019_valid,(solution * (0.307 - wind_lcoe_valid)) * (wind_power_valid * 8760-chec2019_valid) + (solution * (0.307 - wind_lcoe_valid))*chec2019_valid,(solution * (0.307 - wind_lcoe_valid)) * (wind_power_valid * 8760))#m没有运输费
                return np.sum(Q)
            final_population,x,y = multi_objective_differential_evolution(POP_SIZE, max_generations, DIMENSION, CROSSOVER_RATE, MUTATION_RATE)
            # 获取子图的行和列索引
            #row_index = subplot_index // 2
            #col_index = subplot_index % 2
            plot_pareto_front(final_population,axs[row_index][col_index])
            # 绘制折线图
            ax_hist = axs[row_index][col_index].inset_axes([0.6, 0.6, 0.35, 0.35])
            plot_hist_chart(final_population, ax_hist)
            if max_generations == 500:
                plot_line_chart(x , y, axs[5][col_index])
            col_index += 1
    row_index += 1
# 调整布局并显示组图
plt.tight_layout()
plt.show()

In [ ]:

#for ind in final_population:
    #print("Individual:", ind)
    #print("Objectives:", objective_function(ind))

In [ ]:
def plot_hist_chart(final_population, ax):
    # 给定的数组D
    D = np.sum(final_population,axis=0)
    # 创建一个与valid_indices相同长度的数组，用于索引D中的值
    # 由于D的长度小于valid_indices，我们需要重复D中的值
    condition = valid_indices.ravel()
    # 创建一个全0的数组，长度与condition相同
    result = np.zeros_like(condition, dtype=int)
    # 我们可以通过计算valid_indices中True的数量来确定需要重复的次数

    # 计算condition中True的数量
    true_count = np.sum(condition) 
    # 确保D数组有足够的元素来填充True的位置
    if true_count > len(D):
        raise ValueError("D数组中没有足够的元素来填充所有的True位置")
    # 使用numpy的掩码操作来填充True位置
    result[condition] = D[:true_count]
    data = result.reshape(140,309,1)
    D_da = xr.DataArray(
        data=data,
        dims=ds['Wind_Power'].dims,
        coords=ds['Wind_Power'].coords
    )
    # 使用xarray的直方图绘制功能，并且仅绘制非NaN值
    D_da.plot.hist(ax=ax)
    #ax.set_title('Histogram of Count')
    ax.set_xlabel('Count')
    ax.set_ylabel('Selected Number')

In [ ]:
# 给定的数组D
D = np.sum(final_population,axis=0)
# 创建一个与valid_indices相同长度的数组，用于索引D中的值
# 由于D的长度小于valid_indices，我们需要重复D中的值
condition = valid_indices.ravel()
# 创建一个全0的数组，长度与condition相同
result = np.zeros_like(condition, dtype=int)
# 我们可以通过计算valid_indices中True的数量来确定需要重复的次数

# 计算condition中True的数量
true_count = np.sum(condition) 
# 确保D数组有足够的元素来填充True的位置
if true_count > len(D):
    raise ValueError("D数组中没有足够的元素来填充所有的True位置")
# 使用numpy的掩码操作来填充True位置
result[condition] = D[:true_count]
print(result)

In [ ]:
data = result.reshape(140,309,1)

In [ ]:
D_da = xr.DataArray(
    data=data,
    dims=ds['Wind_Power'].dims,
    coords=ds['Wind_Power'].coords
)


In [ ]:
P_da = xr.DataArray(
    data=valid_indices,
    dims=ds['Wind_Power'].dims,
    coords=ds['Wind_Power'].coords
)

In [ ]:
D_da.plot()

In [ ]:
D_da
D_da = D_da.drop_vars(['band', 'number', 'surface'])

In [ ]:
xr.where(D_da>0,D_da,np.nan).plot()

In [ ]:
M = xr.where((D_da>120),1,np.nan)
coordinates1 = M.where(M == 1, drop=True)
print(coordinates1)

In [ ]:
M.plot()

In [ ]:
from shapely.geometry import Point
D_da.name = 'my_data_array'
points_da = D_da.isel(band=0).stack(point=('latitude', 'longitude'))
geodataframe = gpd.GeoDataFrame(points_da.to_dataframe(),geometry=Point(points_da.longitude, points_da.latitude))
geodataframe.to_file('E:/DEdata/Result/Wind_10000_200_140', driver='KML', encoding='utf-8')

In [ ]:
Q = xr.where(D_da,1,np.nan)

In [ ]:
points_da.to_dataframe()

In [ ]:
np.save('/root/Result/End/Solar_y_5000_200.npy',y)
np.save('/root/Result/End/Solar_x_5000_200.npy',x)
np.save('/root/Result/End/Solar_5000_200.npy', final_population)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={'projection': ccrs.Mercator()})

# 设置地图的经纬度范围
ax.set_extent([ds['longitude'].min(), ds['longitude'].max(),
               ds['latitude'].min(), ds['latitude'].max()], crs=ccrs.PlateCarree())

# 添加地理特征
ax.coastlines()

# 绘制数据
# 这里使用 'pcolormesh' 函数来绘制网格数据
plt.pcolormesh(ds['longitude'], ds['latitude'], ds['Wind_LCOE'].isel(band=0), transform=ccrs.PlateCarree())

# 添加颜色条
plt.colorbar(label='Count')

# 设置标题
plt.title('Wind LCOE using WGS84 Coordinates')

# 显示图像
plt.show()

In [ ]:
xr.where((0<ds['Wind_Solar_LCOE'])&(ds['Wind_Solar_LCOE']<0.307),1,np.nan).plot()

In [ ]:
xr.where((ds['Wind_Solar_LCOE']-ds['Wind_LCOE'])<0,(ds['Wind_Solar_LCOE']-ds['Wind_LCOE']),np.nan).plot()

In [ ]:
# 归一化函数
def normalize(arr):
    return (arr - arr.min()) / (arr.max() - arr.min())

# 对 'Wind' 和 'Wind_LCOE' 进行归一化
ds['Solar_normalized'] = normalize(xr.where( (np.isfinite(ds['Solar_Power'])) & (np.isfinite(ds['Solar_LCOE'])) &((ds['Solar_LCOE'] != 0))& ((ds['Solar_Power'] != 0)),ds['Solar_Power'],np.nan))
ds['Solar_LCOE_normalized'] = normalize(xr.where((np.isfinite(ds['Solar_Power'])) & (np.isfinite(ds['Solar_LCOE'])) &((ds['Solar_LCOE'] != 0))& ((ds['Solar_Power'] != 0)),ds['Solar_LCOE'],np.nan))

# 计算 Pearson 相关系数
correlation = xr.corr(ds['Solar_normalized'], ds['Solar_LCOE_normalized'], dim=['longitude','latitude'])

# 绘制散点图
plt.scatter(ds['Solar_normalized'], ds['Solar_LCOE_normalized'])

# 添加标题和轴标签
plt.title('Normalized Solar vs Solar_LCOE Correlation')
plt.xlabel('Normalized Solar Speed')
plt.ylabel('Normalized Solar LCOE')

# 显示相关系数
plt.text(0.1, 0.9, f'Correlation: {correlation.values:.2f}', transform=plt.gca().transAxes)

# 显示图表
plt.show()

In [ ]:
xr.where(ds['Wind_normalized']).plot()